# DNA Sequence Classification using a CNN–LSTM

**Project objective:** classify DNA sequences into their labelled gene-family classes using the Human, Chimpanzee and Dog sequence datasets.

This notebook follows the original project workflow, with a more robust data-loading pipeline, training visualizations, and a corrected multiclass confusion-matrix section.

**Model:** 1D CNN + LSTM  
**Input representation:** one-hot encoded DNA bases (A, C, G, T); N/unknown bases are represented by zeros.  
**Target:** the `class` column supplied in the dataset.

> Run the cells from top to bottom. When prompted, upload `human.txt`, `chimpanzee.txt`, and `dog.txt`. The notebook expects tab-separated files containing `sequence` and `class` columns.

## 1. Import libraries and set reproducibility

In [ ]:
import io
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, accuracy_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Upload the DNA datasets

In [ ]:
from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise ValueError("No files uploaded. Please upload the three dataset .txt files.")

## 3. Read and inspect the uploaded files

In [ ]:
dataframes = {}

for filename, content in uploaded.items():
    if not filename.lower().endswith(".txt"):
        continue

    df = pd.read_csv(io.BytesIO(content), sep="\\t")
    df.columns = df.columns.astype(str).str.strip().str.lower()

    required = {"sequence", "class"}
    if not required.issubset(df.columns):
        raise ValueError(
            f"{filename} must contain columns named 'sequence' and 'class'. "
            f"Found: {df.columns.tolist()}"
        )

    dataframes[filename] = df
    print(f"\\nFile: {filename}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))

if not dataframes:
    raise ValueError("No .txt files were found. Upload human.txt, chimpanzee.txt and dog.txt.")

## 4. Combine, clean and inspect the dataset

In [ ]:
frames = []

for filename, df in dataframes.items():
    name = filename.lower()

    if "human" in name:
        species = "Human"
    elif "dog" in name:
        species = "Dog"
    elif "chimp" in name:
        species = "Chimpanzee"
    else:
        print(f"Skipping unrecognized file: {filename}")
        continue

    part = df[["sequence", "class"]].copy()
    part["species"] = species
    part["sequence"] = part["sequence"].astype(str).str.upper().str.strip()
    part["class"] = pd.to_numeric(part["class"], errors="coerce")

    # Keep valid DNA strings and rows with a valid class label.
    part = part.dropna(subset=["class"])
    part = part[part["sequence"].str.fullmatch(r"[ACGTN]+", na=False)]
    part["class"] = part["class"].astype(int)

    frames.append(part)

if not frames:
    raise ValueError(
        "No recognized datasets were loaded. Check that filenames contain "
        "'human', 'chimp' or 'dog'."
    )

data = pd.concat(frames, ignore_index=True)

if data.empty:
    raise ValueError("The combined dataset is empty after cleaning.")

print("Total valid sequences:", len(data))
print("\\nSequences per species:")
print(data["species"].value_counts())
print("\\nSequences per class:")
print(data["class"].value_counts().sort_index())
print("\\nSequence length statistics:")
display(data["sequence"].str.len().describe())

## 5. Explore class distribution

In [ ]:
class_counts = data["class"].value_counts().sort_index()

plt.figure(figsize=(9, 4))
plt.bar(class_counts.index.astype(str), class_counts.values)
plt.title("Number of DNA Sequences per Class")
plt.xlabel("Class")
plt.ylabel("Number of sequences")
plt.tight_layout()
plt.show()

species_class_counts = pd.crosstab(data["species"], data["class"])
display(species_class_counts)

## 6. Encode DNA sequences

Each nucleotide is represented using one-hot encoding:

- A → `[1, 0, 0, 0]`
- C → `[0, 1, 0, 0]`
- G → `[0, 0, 1, 0]`
- T → `[0, 0, 0, 1]`
- N / padding → `[0, 0, 0, 0]`

The original notebook used a maximum sequence length of 1,000. Longer sequences are therefore truncated to their first 1,000 bases, while shorter sequences are zero-padded. This is a modelling limitation to mention when interpreting results.

In [ ]:
MAX_LEN = 1000
BASE_MAP = {"A": 0, "C": 1, "G": 2, "T": 3}

def encode_dna(sequence, max_len=MAX_LEN):
    encoded = np.zeros((max_len, 4), dtype=np.float32)
    for position, base in enumerate(sequence[:max_len]):
        if base in BASE_MAP:
            encoded[position, BASE_MAP[base]] = 1.0
    return encoded

X = np.stack([encode_dna(seq) for seq in data["sequence"]])
y = data["class"].to_numpy(dtype=np.int32)

CLASS_IDS = np.sort(np.unique(y))
NUM_CLASSES = len(CLASS_IDS)

print("Feature tensor shape:", X.shape)
print("Target shape:", y.shape)
print("Class labels:", CLASS_IDS)

if NUM_CLASSES < 2:
    raise ValueError("At least two different classes are required for classification.")

## 7. Split data into training, validation and test sets

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Training samples:   {len(X_train)} ({len(X_train)/len(X):.0%})")
print(f"Validation samples: {len(X_val)} ({len(X_val)/len(X):.0%})")
print(f"Testing samples:    {len(X_test)} ({len(X_test)/len(X):.0%})")

print("\\nTraining class distribution:")
print(pd.Series(y_train).value_counts().sort_index())

## 8. Calculate class weights

In [ ]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=CLASS_IDS,
    y=y_train
)
class_weights = {int(label): float(weight) for label, weight in zip(CLASS_IDS, weights)}

print("Class weights:")
for label, weight in class_weights.items():
    print(f"Class {label}: {weight:.4f}")

## 9. Build the CNN–LSTM model

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, LSTM, Dropout, Dense

model = Sequential([
    Input(shape=(MAX_LEN, 4)),
    Conv1D(filters=64, kernel_size=7, activation="relu"),
    MaxPooling1D(pool_size=2),
    LSTM(64),
    Dropout(0.30),
    Dense(32, activation="relu"),
    Dense(NUM_CLASSES, activation="softmax")
], name="DNA_CNN_LSTM")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 10. Train the model

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

## 11. Plot training and validation performance

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_df["accuracy"], label="Training accuracy")
plt.plot(history_df["val_accuracy"], label="Validation accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_df["loss"], label="Training loss")
plt.plot(history_df["val_loss"], label="Validation loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 12. Evaluate the model and display the confusion matrices

The first matrix shows raw prediction counts. The second is row-normalized: each row represents the proportion of one actual class assigned to each predicted class. All known class labels are explicitly supplied so that every class appears, even if the model never predicts one of them.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_prob = model.predict(X_test, verbose=0)
y_pred = CLASS_IDS[np.argmax(y_prob, axis=1)]

print(f"Test accuracy: {test_accuracy * 100:.2f}%")
print(f"Test loss: {test_loss:.4f}")

print("\\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    labels=CLASS_IDS,
    target_names=[f"Class {c}" for c in CLASS_IDS],
    zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=CLASS_IDS)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[f"Class {c}" for c in CLASS_IDS]
).plot(
    ax=axes[0],
    cmap="Blues",
    values_format="d",
    colorbar=False,
    xticks_rotation=45
)
axes[0].set_title("Confusion Matrix — Actual Counts")
axes[0].set_xlabel("Predicted class")
axes[0].set_ylabel("Actual class")

row_totals = cm.sum(axis=1, keepdims=True)
cm_normalized = np.divide(
    cm, row_totals,
    out=np.zeros_like(cm, dtype=float),
    where=row_totals != 0
)

im = axes[1].imshow(cm_normalized, cmap="Blues", vmin=0, vmax=1)
axes[1].set_title("Normalized Confusion Matrix")
axes[1].set_xlabel("Predicted class")
axes[1].set_ylabel("Actual class")
axes[1].set_xticks(np.arange(len(CLASS_IDS)))
axes[1].set_yticks(np.arange(len(CLASS_IDS)))
axes[1].set_xticklabels([f"Class {c}" for c in CLASS_IDS], rotation=45, ha="right")
axes[1].set_yticklabels([f"Class {c}" for c in CLASS_IDS])

for i in range(cm_normalized.shape[0]):
    for j in range(cm_normalized.shape[1]):
        axes[1].text(
            j, i, f"{cm_normalized[i, j]:.1%}",
            ha="center", va="center", color="black"
        )

fig.colorbar(im, ax=axes[1], label="Proportion of actual class")
plt.tight_layout()
plt.show()

## 13. Inspect class-wise correct predictions

In [ ]:
correct_by_class = pd.DataFrame({
    "Actual class": CLASS_IDS,
    "Test samples": cm.sum(axis=1),
    "Correct predictions": np.diag(cm),
    "Recall": np.divide(
        np.diag(cm), cm.sum(axis=1),
        out=np.zeros(len(CLASS_IDS), dtype=float),
        where=cm.sum(axis=1) != 0
    )
})
correct_by_class["Recall (%)"] = (correct_by_class.pop("Recall") * 100).round(2)
display(correct_by_class)

## 14. Predict the class of a new DNA sequence

In [ ]:
def predict_family(sequence):
    sequence = str(sequence).upper().strip()

    if not sequence:
        raise ValueError("Please provide a non-empty DNA sequence.")

    invalid = sorted(set(sequence) - set("ACGTN"))
    if invalid:
        raise ValueError(f"Invalid DNA character(s): {invalid}. Use only A, C, G, T or N.")

    encoded = encode_dna(sequence).reshape(1, MAX_LEN, 4)
    probabilities = model.predict(encoded, verbose=0)[0]

    predicted_index = int(np.argmax(probabilities))
    predicted_class = int(CLASS_IDS[predicted_index])

    return {
        "predicted_class": predicted_class,
        "confidence_percent": round(float(probabilities[predicted_index]) * 100, 2),
        "class_probabilities_percent": {
            int(CLASS_IDS[i]): round(float(probabilities[i]) * 100, 2)
            for i in range(len(CLASS_IDS))
        },
        "note": (
            f"Only the first {MAX_LEN} bases are used."
            if len(sequence) > MAX_LEN else "Full sequence used with zero-padding if needed."
        )
    }

# Example only — replace with a sequence from your own use case.
sample_dna = "ATGCGTACGTAGCTAGCTAGCTAGCTAG"
print(predict_family(sample_dna))

## 15. Save the trained model

In [ ]:
model.save("dna_cnn_lstm_model.keras")
print("Model saved as dna_cnn_lstm_model.keras")

# Optional: download the model from Google Colab
from google.colab import files
files.download("dna_cnn_lstm_model.keras")

## 16. Conclusion and limitations

This project implements a multiclass DNA sequence classifier using a hybrid CNN–LSTM architecture. The convolutional layer learns local sequence patterns, while the LSTM layer models sequential relationships. Class weighting is used to reduce the effect of differences in class frequency, and stratified splitting maintains class proportions across the training, validation and test sets.

The final test accuracy, loss, classification report and confusion matrices should be reported from the run of this notebook. Do not assume that a particular accuracy will be achieved: results depend on the uploaded data, training environment and model run.

**Limitations to discuss:**
- The fixed input length of 1,000 bases truncates longer sequences.
- The model uses the supplied class labels; biological interpretation of each class should follow the dataset documentation.
- Accuracy alone is not sufficient for imbalanced multiclass data, so per-class precision, recall, F1-score and the confusion matrix are also reported.
- Further work could compare different sequence lengths, architectures and hyperparameters using validation data.